# 04 – Task 3: Feature Importance

**Question:** Which features contribute most to predicting community membership?

## Method: Permutation Importance

We train a Random Forest on **all features** (community-free + node2vec + struc2vec),
then measure feature importance by **permutation**:

> For each feature, randomly shuffle its values across test nodes and measure
> how much the model's Average Precision drops. A large drop → the feature is important.

This is more reliable than built-in tree importance (which favours high-cardinality features)
because it measures actual impact on prediction quality.

We repeat the permutation `n_repeats=10` times and report the mean drop.

In [ ]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

ROOT = Path("..")
PATH_COMBINED = ROOT / f"data/04_models/{DATASET}_combined.parquet"

print(f"Dataset: {DATASET}")

## Step 1 – Load data

Run `01_data_preparation.ipynb` first.

In [ ]:
df = pd.read_parquet(PATH_COMBINED)

COMMUNITY_FREE  = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
COMMUNITY_AWARE = ["participation", "in_mod_deg"]
N2V_COLS = [c for c in df.columns if c.startswith("emb_node2vec")]
S2V_COLS = [c for c in df.columns if c.startswith("emb_struc2vec")]

# All features together
ALL_FEATURES = COMMUNITY_FREE + N2V_COLS + S2V_COLS

print(f"Nodes    : {len(df)}")
print(f"Features : {len(ALL_FEATURES)}")
print(f"Target   : binary 'target' column")

## Step 2 – Train / test split

In [ ]:
X = df[ALL_FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2023, stratify=y
)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

## Step 3 – Train Random Forest on all features

In [ ]:
forest = RandomForestClassifier(
    n_estimators=1000,
    criterion="gini",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=2023,
)
forest.fit(X_train, y_train)

y_pred_proba = forest.predict_proba(X_test)[:, 1]
print(f"Random Forest baseline on test set:")
print(f"  ROC-AUC          : {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"  Average Precision: {average_precision_score(y_test, y_pred_proba):.4f}")

## Step 4 – Permutation importance

We shuffle each feature 10 times and measure the average drop in Average Precision.

This can take a few minutes on large datasets (we're running 10 × n_features predictions).

In [ ]:
perm_result = permutation_importance(
    forest,
    X_test,
    y_test,
    n_repeats=10,
    random_state=2023,
    scoring="average_precision",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature"       : ALL_FEATURES,
    "importance_mean": perm_result.importances_mean,
    "importance_std" : perm_result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print("Top 20 features by permutation importance:")
importance_df.head(20)

## Step 5 – Visualise

Error bars show standard deviation across 10 permutation repeats.

In [ ]:
TOP_N = 20
top = importance_df.head(TOP_N)

# Assign colors: community-free = blue, node2vec = orange, struc2vec = green
def feature_color(name):
    if name.startswith("emb_node2vec"):  return "tab:orange"
    if name.startswith("emb_struc2vec"): return "tab:green"
    return "tab:blue"

colors = [feature_color(f) for f in top["feature"]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(
    top["feature"][::-1],
    top["importance_mean"][::-1],
    xerr=top["importance_std"][::-1],
    color=colors[::-1],
    capsize=3,
    edgecolor="white",
)
ax.set_xlabel("Drop in Average Precision when feature is permuted")
ax.set_title(f"Permutation Feature Importance – {DATASET}  (top {TOP_N})")

# Legend
from matplotlib.patches import Patch
legend = [
    Patch(color="tab:blue",   label="Community-free graph feature"),
    Patch(color="tab:orange", label="Node2Vec embedding"),
    Patch(color="tab:green",  label="Struc2Vec embedding"),
]
ax.legend(handles=legend, loc="lower right")
plt.tight_layout()
plt.show()

## Step 6 – Summary by feature group

In [ ]:
def feature_group(name):
    if name.startswith("emb_node2vec"):  return "node2vec"
    if name.startswith("emb_struc2vec"): return "struc2vec"
    return "graph_feature"

importance_df["group"] = importance_df["feature"].map(feature_group)

group_summary = (
    importance_df.groupby("group")["importance_mean"]
    .agg(["mean", "sum", "max"])
    .round(5)
    .sort_values("sum", ascending=False)
)
print("Importance by feature group:")
group_summary

In [ ]:
# Save results in long format
PATH_OUT = ROOT / f"data/05_reporting/{DATASET}_task3.parquet"
importance_df.to_parquet(PATH_OUT, index=False)
print(f"Saved to {PATH_OUT}")